In [1]:
import os

import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

In [2]:
#### Load Data
pcs_df_with_kma = pd.read_csv(os.path.join("..", "..", "outputs", "alr", "pcs_with_kma.csv"))
pcs_df_with_kma = pcs_df_with_kma.rename(columns={'kma':'bmus', 'Unnamed: 0':'time'})

pcs_df_with_kma = pcs_df_with_kma.set_index('time')
pcs_df_with_kma.index = pd.to_datetime(pcs_df_with_kma.index)

In [3]:
pcs_df_with_kma

,PC1,PC2,PC3,PC4,PC5,PC6,bmus
time,,,,,,,
1940-01-01,25.647915,15.368420,-4.884146,-3.831472,-8.084204,-3.532147,34
1940-01-02,4.326967,20.108192,-9.036396,11.182830,-21.600030,-2.902122,31
1940-01-03,-3.858448,16.533272,-42.806385,7.233283,-6.602386,4.677763,18
1940-01-04,-23.281990,56.375950,-20.150513,1.056967,6.316501,-0.066660,15
1940-01-05,-27.018059,39.425808,22.984486,1.427371,-2.636404,1.483791,3
...,...,...,...,...,...,...,...
2026-09-26,17.854027,-11.409036,5.071944,14.091280,-4.728758,-0.511893,20
2026-09-27,23.267187,-20.770565,-7.916697,9.281959,-7.978495,3.505971,20
2026-09-28,33.952370,-7.854639,-17.928596,3.261181,-1.171747,-0.972340,22


In [ ]:
import numpy as np
from bluemath_tk.teslakit.statistical import CopulaSimulation

calculated_PCs = {}
simulated_PCs = {}

num_clusters = pcs_df_with_kma.bmus.max() + 1
pc_cols = ["PC1", "PC2", "PC3", "PC4", "PC5", "PC6"]

for cluster in range(0, num_clusters):

    calculated_PCs[f"{cluster}"] = pcs_df_with_kma.loc[pcs_df_with_kma.bmus == cluster, pc_cols].values

    simulated_PCs[f"{cluster}"] = CopulaSimulation(
        U_data=calculated_PCs[f"{cluster}"],
        kernels=["KDE"] * len(pc_cols),
        num_sim=100000,
    )

In [5]:
simulation = xr.open_dataset('/lustre/geocean/WORK/users/alonsoap/projects/TWINPEACS/02_Emulator_Testing/wt_var_testing/outputs/alr/wt_alr_sim_bmus.nc')
bmus_sims  = simulation.evbmus_sims.transpose("n_sim", "time").values.astype(int)   # (n_sim, n_days)
n_sim, n_days = bmus_sims.shape

rng = np.random.default_rng(0)
all_simulated_PCs = np.zeros((n_sim, n_days, len(pc_cols)))

for s in range(n_sim):
    for wt in np.unique(bmus_sims[s]):
        days = np.flatnonzero(bmus_sims[s] == wt)            # days of this record in type wt
        pool = simulated_PCs[f"{wt}"]                        # KDE samples of type wt
        all_simulated_PCs[s, days] = pool[rng.integers(len(pool), size=len(days))]

sim = xr.Dataset(
    {"pcs":  (("seed", "time", "pc"), all_simulated_PCs),
     "bmus": (("seed", "time"), bmus_sims)},
    coords={"seed": range(n_sim), "time": simulation.time.values.astype("datetime64[ns]"), "pc": pc_cols},
)
sim

<xarray.Dataset> Size: 178MB
Dimensions:  (seed: 100, time: 31685, pc: 6)
Coordinates:
  * seed     (seed) int64 800B 0 1 2 3 4 5 6 7 8 ... 91 92 93 94 95 96 97 98 99
  * time     (time) datetime64[ns] 253kB 1940-01-01 1940-01-02 ... 2026-09-30
  * pc       (pc) <U3 72B 'PC1' 'PC2' 'PC3' 'PC4' 'PC5' 'PC6'
Data variables:
    pcs      (seed, time, pc) float64 152MB -9.653 63.23 -17.69 ... 11.15 -4.731
    bmus     (seed, time) int64 25MB 31 31 29 29 5 33 20 ... 32 19 19 19 3 9 9

In [8]:
sim.to_netcdf("../../outputs/model_comp/alr_kde.nc")